In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
import time

API_KEYS = {
    "IN-SO": "tMippMI9OKdcM360xSm1",  # South India
    "NL": "R8LugeduYP4WD8u9zltg",      # Netherlands
    "IN-WE": "EhVgINOIzKKlaOy0iHnr"   # West India
}

REGIONS = {
    "IN-SO": {"lat": 12.9822, "lon": 80.1636},   # Chennai (South India)
    "IN-WE": {"lat": 19.088, "lon": 72.868},     # Mumbai (West India)
    "NL": {"lat": 52.3667, "lon": 4.8945}        # Amsterdam, Netherlands
}

OUT_FILE = "energy_data_historical.csv"
COLLECTION_INTERVAL_MINUTES = 5  # Collect data every 5 minutes

# Fetch historical carbon intensity from ElectricityMap
def fetch_carbon_history(zone, start_time, end_time):
    """Fetch carbon intensity history for a zone"""
    api_key = API_KEYS.get(zone)
    if not api_key:
        print(f"No API key for {zone}")
        return []

    # ElectricityMap historical endpoint
    url = f"https://api.electricitymap.org/v3/carbon-intensity/history?zone={zone}"
    headers = {"auth-token": api_key}

    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code == 200:
            data = r.json()
            history = data.get("history", [])

            # Check date range of returned data
            if history:
                dates = [datetime.fromisoformat(h['datetime'].replace('Z', '+00:00')) for h in history]
                oldest = min(dates)
                newest = max(dates)
                hours_available = (newest - oldest).total_seconds() / 3600
                print(f"✓ Fetched {len(history)} carbon records for {zone}")
                print(f"  Data range: {oldest.strftime('%Y-%m-%d %H:%M')} to {newest.strftime('%Y-%m-%d %H:%M')} ({hours_available:.0f} hours)")
            else:
                print(f"⚠ No carbon data returned for {zone}")

            return history
        else:
            print(f"✗ Carbon fetch failed for {zone}: {r.status_code} - {r.text}")
    except Exception as e:
        print(f"✗ Carbon fetch error for {zone}: {str(e)}")

    return []

# Fetch historical weather from Open-Meteo
def fetch_weather_history(lat, lon, start_date, end_date):
    """Fetch historical weather data"""
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}"
        f"&hourly=cloudcover,windspeed_10m,temperature_2m"
        f"&start_date={start_date}&end_date={end_date}"
        f"&timezone=UTC"
    )

    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            hourly = data.get('hourly', {})
            times = hourly.get('time', [])
            clouds = hourly.get('cloudcover', [])
            winds = hourly.get('windspeed_10m', [])
            temps = hourly.get('temperature_2m', [])

            weather_dict = {}
            for i, t in enumerate(times):
                timestamp = datetime.fromisoformat(t.replace('Z', '+00:00'))
                weather_dict[timestamp] = {
                    'solar_cloud_pct': 100 - clouds[i] if i < len(clouds) and clouds[i] is not None else None,
                    'wind_speed': winds[i] if i < len(winds) else None,
                    'temperature': temps[i] if i < len(temps) else None
                }

            print(f"✓ Fetched {len(weather_dict)} weather records for lat={lat}, lon={lon}")
            return weather_dict
        else:
            print(f"✗ Weather fetch failed: {r.status_code}")
    except Exception as e:
        print(f"✗ Weather fetch error: {str(e)}")

    return {}

# Collect historical data
def collect_historical(hours_back=24, interval_minutes=None):
    """Collect historical data for the past N hours at specified intervals

    Args:
        hours_back: Total hours to collect (default 24)
        interval_minutes: Minutes between data points (default uses COLLECTION_INTERVAL_MINUTES)
    """
    if interval_minutes is None:
        interval_minutes = COLLECTION_INTERVAL_MINUTES

    end = datetime.now(timezone.utc)
    start = end - timedelta(hours=hours_back)

    start_date = start.strftime('%Y-%m-%d')
    end_date = end.strftime('%Y-%m-%d')

    expected_points = int((hours_back * 60) / interval_minutes)

    print(f"\n=== Collecting data from {start} to {end} UTC ===")
    print(f"=== Interval: Every {interval_minutes} minute(s) ===")
    print(f"=== Expected data points per region: ~{expected_points} ===\n")

    all_rows = []

    for zone, coords in REGIONS.items():
        print(f"\n--- Processing {zone} ---")

        # Fetch carbon history
        carbon_history = fetch_carbon_history(zone, start, end)

        # Fetch weather history
        weather_history = fetch_weather_history(
            coords["lat"],
            coords["lon"],
            start_date,
            end_date
        )

        # Create carbon lookup dictionary
        carbon_dict = {}
        for record in carbon_history:
            try:
                dt_str = record['datetime'].replace('Z', '+00:00')
                dt = datetime.fromisoformat(dt_str)
                # Ensure timezone aware
                if dt.tzinfo is None:
                    dt = dt.replace(tzinfo=timezone.utc)
                carbon_dict[dt] = record.get('carbonIntensity')
            except Exception as e:
                continue

        # Merge data by aligning timestamps at 5-minute intervals
        # Note: API data is hourly, so we'll interpolate/repeat values for 5-min intervals
        current = start.replace(minute=0, second=0, microsecond=0)

        while current <= end:
            # Find closest carbon intensity (hourly data)
            carbon = None
            min_diff = float('inf')
            for dt, ci in carbon_dict.items():
                diff = abs((dt - current).total_seconds())
                if diff < min_diff and diff < 3600:  # within 1 hour
                    min_diff = diff
                    carbon = ci

            # Get closest weather data (hourly data)
            weather = {}
            min_diff = float('inf')
            for wt, wdata in weather_history.items():
                # Make weather timestamp timezone-aware if needed
                if wt.tzinfo is None:
                    wt = wt.replace(tzinfo=timezone.utc)
                diff = abs((wt - current).total_seconds())
                if diff < min_diff and diff < 3600:  # within 1 hour
                    min_diff = diff
                    weather = wdata

            row = {
                "timestamp": current,
                "region": zone,
                "carbon_intensity": carbon,
                "solar_cloud_pct": weather.get('solar_cloud_pct'),
                "wind_speed": weather.get('wind_speed'),
                "temperature": weather.get('temperature')
            }
            all_rows.append(row)

            current += timedelta(minutes=interval_minutes)  # 5-minute intervals

        # Rate limiting - be nice to APIs
        time.sleep(1)

    # Save to CSV
    df = pd.DataFrame(all_rows)
    df.to_csv(OUT_FILE, index=False)

    print(f"\n=== Summary ===")
    print(f"Total rows: {len(df)}")
    print(f"Data points per region: {len(df) // len(REGIONS)}")
    print(f"Rows with carbon data: {df['carbon_intensity'].notna().sum()}")
    print(f"Rows with weather data: {df['solar_cloud_pct'].notna().sum()}")

    # Check data coverage by region
    print(f"\nData coverage by region:")
    for zone in REGIONS.keys():
        zone_data = df[df['region'] == zone]
        carbon_pct = (zone_data['carbon_intensity'].notna().sum() / len(zone_data)) * 100
        print(f"  {zone}: {carbon_pct:.1f}% coverage ({zone_data['carbon_intensity'].notna().sum()}/{len(zone_data)} points)")

    print(f"\nSaved to {OUT_FILE}")

    # Show sample - first and last few rows
    print("\nFirst 5 data points:")
    print(df.head(5))
    print("\nLast 5 data points:")
    print(df.tail(5))

    return df

if __name__ == "__main__":
    # Collect 24 hours of data at 5-minute intervals
    # This will create ~288 data points per region (24 hours * 12 five-minute intervals per hour)
    # Total: ~864 rows (288 * 3 regions)
    print("📊 Creating high-resolution dataset with 5-minute intervals")
    print("⚠️  Note: API provides hourly data, so values will repeat within each hour\n")

    df = collect_historical(hours_back=24, interval_minutes=5)

📊 Creating high-resolution dataset with 5-minute intervals
⚠️  Note: API provides hourly data, so values will repeat within each hour


=== Collecting data from 2025-10-07 16:39:53.523436+00:00 to 2025-10-08 16:39:53.523436+00:00 UTC ===
=== Interval: Every 5 minute(s) ===
=== Expected data points per region: ~288 ===


--- Processing IN-SO ---
✓ Fetched 24 carbon records for IN-SO
  Data range: 2025-10-07 17:00 to 2025-10-08 16:00 (23 hours)
✓ Fetched 48 weather records for lat=12.9822, lon=80.1636

--- Processing IN-WE ---
✓ Fetched 24 carbon records for IN-WE
  Data range: 2025-10-07 17:00 to 2025-10-08 16:00 (23 hours)
✓ Fetched 48 weather records for lat=19.088, lon=72.868

--- Processing NL ---
✓ Fetched 24 carbon records for NL
  Data range: 2025-10-07 17:00 to 2025-10-08 16:00 (23 hours)
✓ Fetched 48 weather records for lat=52.3667, lon=4.8945

=== Summary ===
Total rows: 888
Data points per region: 296
Rows with carbon data: 885
Rows with weather data: 888

Data coverage by re

In [ ]:
df.to_csv(OUT_FILE, index=False)
OUT_FILE = "energy_data_historical.csv"


In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
import time
import os

# ---------------- CONFIG ----------------
API_KEYS = {
    "IN-SO": "tMippMI9OKdcM360xSm1",  # South India
    "NL": "R8LugeduYP4WD8u9zltg",      # Netherlands
    "IN-WE": "EhVgINOIzKKlaOy0iHnr"   # West India
}

REGIONS = {
    "IN-SO": {"lat": 12.9822, "lon": 80.1636},   # Chennai
    "IN-WE": {"lat": 19.088, "lon": 72.868},     # Mumbai
    "NL": {"lat": 52.3667, "lon": 4.8945}        # Amsterdam
}

OUT_FILE = "energy_data_historical.csv"
COLLECTION_INTERVAL_MINUTES = 5  # 5-minute intervals

# ----------------------------------------

# --- Functions to fetch data ---

def fetch_carbon_history(zone, start_time, end_time):
    """Fetch carbon intensity history for a zone from ElectricityMap"""
    api_key = API_KEYS.get(zone)
    if not api_key:
        print(f"No API key for {zone}")
        return []

    url = f"https://api.electricitymap.org/v3/carbon-intensity/history?zone={zone}"
    headers = {"auth-token": api_key}

    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code == 200:
            data = r.json()
            history = data.get("history", [])
            print(f"✓ Fetched {len(history)} carbon records for {zone}")
            return history
        else:
            print(f"✗ Carbon fetch failed for {zone}: {r.status_code}")
    except Exception as e:
        print(f"✗ Carbon fetch error for {zone}: {str(e)}")
    return []

def fetch_weather_history(lat, lon, start_date, end_date):
    """Fetch historical weather data from Open-Meteo"""
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}"
        f"&hourly=cloudcover,windspeed_10m,temperature_2m"
        f"&start_date={start_date}&end_date={end_date}"
        f"&timezone=UTC"
    )

    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            hourly = data.get('hourly', {})
            times = hourly.get('time', [])
            clouds = hourly.get('cloudcover', [])
            winds = hourly.get('windspeed_10m', [])
            temps = hourly.get('temperature_2m', [])

            weather_dict = {}
            for i, t in enumerate(times):
                timestamp = datetime.fromisoformat(t.replace('Z', '+00:00'))
                weather_dict[timestamp] = {
                    'solar_cloud_pct': 100 - clouds[i] if i < len(clouds) and clouds[i] is not None else None,
                    'wind_speed': winds[i] if i < len(winds) else None,
                    'temperature': temps[i] if i < len(temps) else None
                }
            print(f"✓ Fetched {len(weather_dict)} weather records for lat={lat}, lon={lon}")
            return weather_dict
        else:
            print(f"✗ Weather fetch failed: {r.status_code}")
    except Exception as e:
        print(f"✗ Weather fetch error: {str(e)}")
    return {}

# --- Main collection function ---

def collect_historical_append(hours_back=24, interval_minutes=None):
    if interval_minutes is None:
        interval_minutes = COLLECTION_INTERVAL_MINUTES

    end = datetime.now(timezone.utc)
    start = end - timedelta(hours=hours_back)

    start_date = start.strftime('%Y-%m-%d')
    end_date = end.strftime('%Y-%m-%d')

    all_rows = []

    # Load existing CSV if exists
    if os.path.exists(OUT_FILE):
        df_existing = pd.read_csv(OUT_FILE, parse_dates=['timestamp'])
        print(f"Existing CSV found: {len(df_existing)} rows")
    else:
        df_existing = pd.DataFrame()
        print("No existing CSV found, creating new dataset")

    for zone, coords in REGIONS.items():
        print(f"\n--- Processing {zone} ---")

        # Determine last timestamp in existing CSV for this zone
        last_ts = None
        if not df_existing.empty:
            zone_data = df_existing[df_existing['region'] == zone]
            if not zone_data.empty:
                last_ts = zone_data['timestamp'].max()
                last_ts = pd.to_datetime(last_ts).to_pydatetime()
                last_ts = last_ts.replace(tzinfo=timezone.utc)

        # Adjust start to last timestamp + interval
        adjusted_start = start
        if last_ts and last_ts > start:
            adjusted_start = last_ts + timedelta(minutes=interval_minutes)

        if adjusted_start >= end:
            print(f"No new data to fetch for {zone}")
            continue

        # Fetch carbon and weather data
        carbon_history = fetch_carbon_history(zone, adjusted_start, end)
        weather_history = fetch_weather_history(coords["lat"], coords["lon"], start_date, end_date)

        # Create carbon lookup dictionary
        carbon_dict = {}
        for record in carbon_history:
            try:
                dt_str = record['datetime'].replace('Z', '+00:00')
                dt = datetime.fromisoformat(dt_str)
                if dt.tzinfo is None:
                    dt = dt.replace(tzinfo=timezone.utc)
                carbon_dict[dt] = record.get('carbonIntensity')
            except Exception:
                continue

        # Merge data by interval
        current = adjusted_start.replace(second=0, microsecond=0)
        while current <= end:
            # Closest carbon intensity
            carbon = None
            min_diff = float('inf')
            for dt, ci in carbon_dict.items():
                diff = abs((dt - current).total_seconds())
                if diff < min_diff and diff < 3600:
                    min_diff = diff
                    carbon = ci

            # Closest weather
            weather = {}
            min_diff = float('inf')
            for wt, wdata in weather_history.items():
                if wt.tzinfo is None:
                    wt = wt.replace(tzinfo=timezone.utc)
                diff = abs((wt - current).total_seconds())
                if diff < min_diff and diff < 3600:
                    min_diff = diff
                    weather = wdata

            row = {
                "timestamp": current,
                "region": zone,
                "carbon_intensity": carbon,
                "solar_cloud_pct": weather.get('solar_cloud_pct'),
                "wind_speed": weather.get('wind_speed'),
                "temperature": weather.get('temperature')
            }
            all_rows.append(row)

            current += timedelta(minutes=interval_minutes)

        time.sleep(1)  # polite delay for API

    # Append to CSV
    df_new = pd.DataFrame(all_rows)
    if os.path.exists(OUT_FILE):
        df_new.to_csv(OUT_FILE, mode='a', header=False, index=False)
        print(f"\nAppended {len(df_new)} new rows to {OUT_FILE}")
    else:
        df_new.to_csv(OUT_FILE, index=False)
        print(f"\nSaved {len(df_new)} rows to new CSV {OUT_FILE}")

    return df_new

# ---------------- RUN ----------------
if __name__ == "__main__":
    print("📊 Appending fresh data to existing CSV")
    df_fresh = collect_historical_append(hours_back=24, interval_minutes=5)


📊 Appending fresh data to existing CSV
Existing CSV found: 3423 rows

--- Processing IN-SO ---
✓ Fetched 24 carbon records for IN-SO
✓ Fetched 48 weather records for lat=12.9822, lon=80.1636

--- Processing IN-WE ---
✓ Fetched 24 carbon records for IN-WE
✓ Fetched 48 weather records for lat=19.088, lon=72.868

--- Processing NL ---
✓ Fetched 24 carbon records for NL
✓ Fetched 48 weather records for lat=52.3667, lon=4.8945

Appended 867 new rows to energy_data_historical.csv


In [ ]:
!pip install pymoo

In [ ]:
# ------------------ Interactive Cloud Job Scheduler with Future Prediction ------------------

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import IntegerRandomSampling
from pymoo.optimize import minimize
from datetime import datetime, timedelta

# ------------------ Step 1: User Input Collection ------------------

print("=" * 70)
print("🌍 CLOUD JOB SCHEDULER - FUTURE PREDICTION & OPTIMIZATION")
print("=" * 70)
print()

# Get number of jobs
while True:
    try:
        N_JOBS = int(input("Enter the number of cloud jobs to schedule: "))
        if N_JOBS > 0:
            break
        else:
            print("❌ Please enter a positive number.")
    except ValueError:
        print("❌ Invalid input. Please enter a number.")

print()

# Get prediction window
print("📅 Future Scheduling Window")
print("-" * 70)
while True:
    try:
        forecast_days = int(input("How many days ahead to schedule? (e.g., 7): "))
        if forecast_days > 0:
            break
        else:
            print("❌ Please enter a positive number.")
    except ValueError:
        print("❌ Invalid input. Please enter a number.")

print()

# Get objective weights
print("📊 Define Objective Weights (0.0 to 1.0)")
print("   Higher weight = Higher priority in optimization")
print("-" * 70)

while True:
    try:
        weight_carbon = float(input("Weight for Carbon Intensity (e.g., 0.7): "))
        if 0 <= weight_carbon <= 1:
            break
        else:
            print("❌ Please enter a value between 0 and 1.")
    except ValueError:
        print("❌ Invalid input. Please enter a decimal number.")

# Automatically calculate renewable weight as complement
weight_renewable = 1.0 - weight_carbon
print(f"   → Renewable Energy weight auto-set to: {weight_renewable:.2f}")

print()

# Get load balancing weight
print("⚖️  Load Balancing Configuration")
print("-" * 70)

while True:
    try:
        weight_load = float(input("Weight for Load Balancing Penalty (e.g., 0.1): "))
        if 0 <= weight_load <= 1:
            break
        else:
            print("❌ Please enter a value between 0 and 1.")
    except ValueError:
        print("❌ Invalid input. Please enter a decimal number.")

print()

# ------------------ Step 2: Load Historical Data ------------------

print("📁 Loading historical dataset...")
CSV_FILE = "energy_data_historical.csv"
df_historical = pd.read_csv(CSV_FILE, parse_dates=['timestamp'])

# Sort and forward-fill missing values
df_historical = df_historical.sort_values(['region', 'timestamp'])
df_historical[['carbon_intensity', 'solar_cloud_pct', 'wind_speed', 'temperature']] = df_historical.groupby('region')[
    ['carbon_intensity', 'solar_cloud_pct', 'wind_speed', 'temperature']].ffill()

print(f"✅ Loaded {len(df_historical)} historical records")
print(f"   Date range: {df_historical['timestamp'].min()} to {df_historical['timestamp'].max()}")
print()

# ------------------ Step 3: Build Forecasting Models ------------------

print("🔮 Training forecasting models...")

# Feature engineering for time series
df_historical['hour'] = df_historical['timestamp'].dt.hour
df_historical['day_of_week'] = df_historical['timestamp'].dt.dayofweek
df_historical['month'] = df_historical['timestamp'].dt.month
df_historical['day_of_year'] = df_historical['timestamp'].dt.dayofyear

# Create lagged features (previous values)
for col in ['carbon_intensity', 'solar_cloud_pct', 'wind_speed', 'temperature']:
    df_historical[f'{col}_lag1'] = df_historical.groupby('region')[col].shift(1)

df_historical = df_historical.dropna()

# Features for prediction
feature_cols = ['hour', 'day_of_week', 'month', 'day_of_year',
                'carbon_intensity_lag1', 'solar_cloud_pct_lag1',
                'wind_speed_lag1', 'temperature_lag1']

target_cols = ['carbon_intensity', 'solar_cloud_pct', 'wind_speed', 'temperature']

# Train models for each target and region
models = {}
regions = df_historical['region'].unique()

for region in regions:
    df_region = df_historical[df_historical['region'] == region]
    models[region] = {}

    for target in target_cols:
        X = df_region[feature_cols]
        y = df_region[target]

        model = RandomForestRegressor(n_estimators=50, random_state=42, max_depth=10)
        model.fit(X, y)
        models[region][target] = model

print(f"✅ Trained models for {len(regions)} regions")
print()

# ------------------ Step 4: Generate Future Predictions ------------------

print(f"🔮 Predicting conditions for next {forecast_days} days...")

# Generate future timestamps
last_timestamp = df_historical['timestamp'].max()
future_timestamps = pd.date_range(
    start=last_timestamp + timedelta(minutes=5),
    periods=forecast_days * 288,  # 288 = 5-min intervals per day
    freq='5min'
)

# Create future dataframe
future_data = []

for region in regions:
    # Get last known values for this region
    last_values = df_historical[df_historical['region'] == region].iloc[-1]

    for ts in future_timestamps:
        # Create features
        features = {
            'timestamp': ts,
            'region': region,
            'hour': ts.hour,
            'day_of_week': ts.dayofweek,
            'month': ts.month,
            'day_of_year': ts.dayofyear,
            'carbon_intensity_lag1': last_values['carbon_intensity'],
            'solar_cloud_pct_lag1': last_values['solar_cloud_pct'],
            'wind_speed_lag1': last_values['wind_speed'],
            'temperature_lag1': last_values['temperature']
        }

        # Predict each target
        for target in target_cols:
            X_pred = pd.DataFrame([features])[feature_cols]
            prediction = models[region][target].predict(X_pred)[0]
            features[target] = prediction

        future_data.append(features)

        # Update lag values for next iteration
        last_values = pd.Series(features)

df_future = pd.DataFrame(future_data)

print(f"✅ Generated {len(df_future)} future time slots")
print(f"   Future date range: {df_future['timestamp'].min()} to {df_future['timestamp'].max()}")
print()

# ------------------ Step 5: Normalize Future Data ------------------

print("📊 Normalizing predicted data...")

# Use same scaler fit on historical data
scaler = MinMaxScaler()
scaler.fit(df_historical[['carbon_intensity', 'solar_cloud_pct', 'wind_speed', 'temperature']])

df_future[['carbon_norm', 'solar_norm', 'wind_norm', 'temp_norm']] = scaler.transform(
    df_future[['carbon_intensity', 'solar_cloud_pct', 'wind_speed', 'temperature']]
)

# Compute combined renewable availability
df_future['renewable_norm'] = (df_future['solar_norm'] + df_future['wind_norm']) / 2

# Assign unique indices
df_future = df_future.reset_index(drop=True)
df_future['slot_index'] = df_future.index

# Calculate load balancing parameters
region_counts = df_future.groupby('region')['slot_index'].count().to_dict()
n_regions = len(region_counts)
max_jobs_per_region = N_JOBS // n_regions + (1 if N_JOBS % n_regions != 0 else 0)

print(f"   Number of regions: {n_regions}")
print(f"   Max jobs per region: {max_jobs_per_region}")
print()

# ------------------ Step 6: Define NSGA-II Problem ------------------

class CloudSchedulingProblem(Problem):
    def __init__(self, df, n_jobs, max_jobs_per_region,
                 weight_carbon, weight_renewable, weight_load):
        self.df = df.reset_index(drop=True)
        self.n_jobs = n_jobs
        self.max_jobs_per_region = max_jobs_per_region
        self.weight_carbon = weight_carbon
        self.weight_renewable = weight_renewable
        self.weight_load = weight_load

        n_var = n_jobs
        super().__init__(n_var=n_var, n_obj=2, n_constr=1,
                         xl=0, xu=len(df)-1, type_var=int)

    def _evaluate(self, X, out, *args, **kwargs):
        F = []
        G = []

        for sol in X:
            sol_int = np.array(np.round(sol), dtype=int)

            # Objective 1: Minimize Carbon
            carbon = np.array([self.df.loc[i, 'carbon_norm'] for i in sol_int])
            obj_carbon = self.weight_carbon * carbon.mean()

            # Objective 2: Maximize Renewable
            renewable = np.array([self.df.loc[i, 'renewable_norm'] for i in sol_int])
            obj_renewable = -self.weight_renewable * renewable.mean()

            # Constraint: No duplicates
            duplicates = len(sol_int) - len(np.unique(sol_int))

            # Load Balancing Penalty
            region_counts = self.df.loc[sol_int, 'region'].value_counts()
            load_penalty = sum(max(0, count - self.max_jobs_per_region)
                             for count in region_counts)

            obj_carbon += self.weight_load * load_penalty

            F.append([obj_carbon, obj_renewable])
            G.append([duplicates])

        out["F"] = np.array(F)
        out["G"] = np.array(G)

# ------------------ Step 7: Run NSGA-II Optimization ------------------

print("🚀 Starting optimization...")
print(f"   Algorithm: NSGA-II")
print(f"   Population size: 100")
print(f"   Generations: 200")
print()

problem = CloudSchedulingProblem(
    df_future,
    n_jobs=N_JOBS,
    max_jobs_per_region=max_jobs_per_region,
    weight_carbon=weight_carbon,
    weight_renewable=weight_renewable,
    weight_load=weight_load
)

algorithm = NSGA2(
    pop_size=100,
    sampling=IntegerRandomSampling(),
    crossover=SBX(prob=0.9, eta=15, vtype=float, repair=None),
    mutation=PM(prob=0.1, eta=20, vtype=float, repair=None),
    eliminate_duplicates=True
)

res = minimize(
    problem,
    algorithm,
    ('n_gen', 200),
    seed=42,
    verbose=False
)

print("✅ Optimization complete!")
print()

# ------------------ Step 8: Extract Best Schedule ------------------

best_solution_int = np.array(np.round(res.X[0]), dtype=int)

assert len(best_solution_int) == N_JOBS, f"Solution mismatch: {len(best_solution_int)} vs {N_JOBS}"

scheduled_slots = df_future.loc[best_solution_int,
                         ['timestamp', 'region', 'carbon_intensity',
                          'solar_cloud_pct', 'wind_speed', 'temperature']].copy()

scheduled_slots['job_id'] = range(1, N_JOBS + 1)

# Sort by timestamp for better readability
scheduled_slots = scheduled_slots.sort_values('timestamp').reset_index(drop=True)

# Calculate metrics
avg_carbon = scheduled_slots['carbon_intensity'].mean()
avg_solar = scheduled_slots['solar_cloud_pct'].mean()
avg_wind = scheduled_slots['wind_speed'].mean()
region_distribution = scheduled_slots['region'].value_counts()

# ------------------ Step 9: Display Results ------------------

print("=" * 70)
print("📊 OPTIMIZATION RESULTS (FUTURE SCHEDULE)")
print("=" * 70)
print()

print("🎯 Objectives:")
print(f"   Carbon Intensity (weighted): {res.F[0][0]:.4f}")
print(f"   Renewable Energy (weighted): {-res.F[0][1]:.4f}")
print()

print("📈 Predicted Schedule Metrics:")
print(f"   Average Carbon Intensity: {avg_carbon:.2f} gCO2/kWh")
print(f"   Average Solar Cloud Cover: {avg_solar:.1f}%")
print(f"   Average Wind Speed: {avg_wind:.1f} m/s")
print()

print("🌍 Regional Distribution:")
for region, count in region_distribution.items():
    percentage = (count / N_JOBS) * 100
    print(f"   {region}: {count} jobs ({percentage:.1f}%)")
print()

print("📋 Sample Schedule (first 10 jobs):")
print(scheduled_slots.head(10).to_string(index=False))
print()

# ------------------ Step 10: Save Results ------------------

output_file = "cloud_schedule_future_optimal.csv"
scheduled_slots.to_csv(output_file, index=False)
print(f"💾 Saved optimal future schedule to: {output_file}")
print()
print("=" * 70)

🌍 CLOUD JOB SCHEDULER - FUTURE PREDICTION & OPTIMIZATION

Enter the number of cloud jobs to schedule: 100

📅 Future Scheduling Window
----------------------------------------------------------------------
How many days ahead to schedule? (e.g., 7): 1

📊 Define Objective Weights (0.0 to 1.0)
   Higher weight = Higher priority in optimization
----------------------------------------------------------------------
Weight for Carbon Intensity (e.g., 0.7): 0.6
   → Renewable Energy weight auto-set to: 0.40

⚖️  Load Balancing Configuration
----------------------------------------------------------------------
Weight for Load Balancing Penalty (e.g., 0.1): 0.4

📁 Loading historical dataset...
✅ Loaded 1689 historical records
   Date range: 2025-10-07 16:00:00+00:00 to 2025-10-09 14:50:00+00:00

🔮 Training forecasting models...
✅ Trained models for 3 regions

🔮 Predicting conditions for next 1 days...
✅ Generated 864 future time slots
   Future date range: 2025-10-09 14:55:00+00:00 to 2025-10-